# LoRA / QLoRA Fine-Tuning (Local Jupyter Version)


## 1. Environment check

In [1]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No CUDA GPU detected.")


CUDA available: True
GPU: NVIDIA GeForce RTX 3080
VRAM: 12.9 GB


## 2. Install dependencies


In [2]:
# pip install transformers peft trl bitsandbytes accelerate datasets python-dotenv


## 3. Hugging Face authentication


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.


## 4. Load the base model (4-bit quantized)

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "meta-llama/Llama-3.2-1B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

## 5. Configure LoRA


In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


## 6. Load / prepare the dataset


In [14]:
from datasets import load_dataset

# 1. Load FinGPT sentiment dataset
dataset = load_dataset("FinGPT/fingpt-sentiment-train")

# 2. Format prompt and append EOS token at the end of output
def format_prompts(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        formatted_text = (
            f"Instruction:\n{inst}\n\n"
            f"Input:\n{inp}\n\n"
            f"Answer:\n{out}{tokenizer.eos_token}"
        )
        texts.append(formatted_text)
    return {"text": texts}

# 3. Map formatting to dataset
dataset = dataset.map(format_prompts, batched=True)

# Verify 'text' column
print("Columns:", dataset["train"].column_names)
print("\nSample formatted text:\n", dataset["train"][0]["text"])

Map:   0%|          | 0/76772 [00:00<?, ? examples/s]

Columns: ['input', 'output', 'instruction', 'text']

Sample formatted text:
 Instruction:
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

Input:
Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .

Answer:
neutral<|end_of_text|>


## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./lora-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,       # ~10x a typical full fine-tune LR, per the 2026 guidance above
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
)

trainer.train()


In [16]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./lora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=256,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/76772 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/76772 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/76772 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/76772 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/76772 [00:00<?, ? examples/s]

Step,Training Loss
10,2.434148
20,1.778478
30,1.709214
40,1.653555
50,1.489062
60,1.496422
70,1.462727
80,1.413423
90,1.466573
100,1.498715


TrainOutput(global_step=4799, training_loss=1.0219103874179516, metrics={'train_runtime': 16467.3662, 'train_samples_per_second': 4.662, 'train_steps_per_second': 0.291, 'total_flos': 3.2638227424518144e+16, 'train_loss': 1.0219103874179516, 'entropy': 0.778037235818126, 'num_tokens': 4744641.0, 'mean_token_accuracy': 0.8532465895016988, 'epoch': 1.0})

## 8. Compare outputs before/after fine-tuning

Generate from the same prompt(s) and compare qualitatively — this comparison is the actual proof of understanding, not just a successful training run.


In [ ]:
prompt = "Replace with a domain-specific test prompt"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0], skip_special_tokens=True))


In [19]:
import re

model.config.use_cache = True

prompt = """Instruction:
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

Input:
Tesla beats quarterly revenue expectations as electric vehicle deliveries hit record highs.

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

prompt_length = inputs["input_ids"].shape[1]
raw_response = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True).strip()

# Clean up: stop immediately at the word "Instruction" or extract the target label
clean_response = raw_response.split("Instruction")[0].strip()

# Or match the exact sentiment class
match = re.search(r'\b(positive|negative|neutral)\b', clean_response, re.IGNORECASE)
final_sentiment = match.group(0).lower() if match else clean_response

print("--- Generated Output ---")
print(final_sentiment)

--- Generated Output ---
positive


## 9. Save Trained LoRA Adapter

In [20]:
# Save LoRA adapter checkpoints
adapter_dir = "./lora-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved LoRA adapter to {adapter_dir}")

Saved LoRA adapter to ./lora-adapter


## 10. Push Adapter to Hugging Face Hub

In [4]:
from huggingface_hub import HfApi

write_token = os.getenv("WRITE_TOKEN") or os.getenv("HF_TOKEN")
api = HfApi(token=write_token)

repo_id = "Yutakuroda/llama-3.2-1b-fingpt-sentiment"

api.upload_folder(
    folder_path="./lora-adapter",
    repo_id=repo_id,
    repo_type="model"
)

print(f"Successfully uploaded LoRA adapter to: https://huggingface.co/{repo_id}")

No files have been modified since last commit. Skipping to prevent empty commit.


Successfully uploaded LoRA adapter to: https://huggingface.co/Yutakuroda/llama-3.2-1b-fingpt-sentiment
